# FreshPick Image Data Preparation


## 1. Setup and source download



In [ ]:
# Uncomment if these packages are missing.
# %pip install numpy pandas pillow

from pathlib import Path
from collections import Counter
from io import BytesIO
import hashlib
import json
import re
import urllib.request
import zipfile
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageFilter

BASE = Path.cwd()
if (BASE / "agrifreshnet_data_preparation").is_dir():
    BASE = BASE / "agrifreshnet_data_preparation"
SOURCE = BASE / "source" / "Processed Data.zip"
OUT = BASE / "prepared_dataset"
SOURCE.parent.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)
URL = "https://data.mendeley.com/public-files/datasets/42m5tb7yv9/files/7b216853-8f3b-47fd-b1ae-df88a367a1d1/file_downloaded"
EXPECTED_SHA256 = "6607a48bf2817f9dfc8d4a1c32ce663c1d8308b4b1251e9bf65f074fcac53f11"
if not SOURCE.exists():
    partial = SOURCE.with_suffix(".zip.partial")
    urllib.request.urlretrieve(URL, partial)
    partial.replace(SOURCE)
def file_hash(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()
archive_hash = file_hash(SOURCE)
assert archive_hash == EXPECTED_SHA256, "Source archive checksum mismatch; download again."
print("Source checksum verified:", archive_hash)


Source checksum verified: 6607a48bf2817f9dfc8d4a1c32ce663c1d8308b4b1251e9bf65f074fcac53f11


## 2. Inventory and annotation wrangling

The archive inventory determines the actual image count. Inconsistent capitalization and spacing in folder labels are standardized into produce, freshness, and class fields. The original folder and filename remain in the audit trail.


In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
archive = zipfile.ZipFile(SOURCE)
assert archive.testzip() is None, "Archive CRC check failed."
entries = sorted(n for n in archive.namelist()
                 if not n.endswith("/") and Path(n).suffix.lower() in IMAGE_EXTENSIONS)
folders = sorted({Path(n).parent.name for n in entries})
def parse_label(folder):
    match = re.fullmatch(r"(.+?)\s*\(\s*(\d+)\s*-\s*(\d+)\s*\)", folder)
    if not match:
        raise ValueError(f"Unrecognized folder: {folder}")
    text = re.sub(r"[_\s]+", " ", match[1].strip()).lower()
    for prefix, freshness in [("semi fresh ", "semi_fresh"), ("fresh ", "fresh"), ("rotten ", "rotten")]:
        if text.startswith(prefix):
            produce = text[len(prefix):].strip()
            return {"class_name": f"{freshness}_{produce}", "produce": produce,
                    "freshness": freshness, "publisher_day_min": int(match[2]),
                    "publisher_day_max": int(match[3])}
    raise ValueError(f"Unrecognized freshness: {folder}")
labels = {folder: parse_label(folder) for folder in folders}
class_names = sorted({v["class_name"] for v in labels.values()})
class_to_id = {name: i for i, name in enumerate(class_names)}
class_table = pd.DataFrame([{**v, "class_id": class_to_id[v["class_name"]],
                            "source_folder": k} for k, v in labels.items()])
class_table.to_csv(OUT / "class_map.csv", index=False)
print("Archive images:", len(entries), "Classes:", len(class_names))
print(pd.Series(Counter(Path(n).parent.name for n in entries)).to_string())


Archive images: 14160 Classes: 24
Fresh Banana(1-4)                590
Fresh Bittermelon(1-3)           590
Fresh Cucumber(1-6)              590
Fresh Orange(1-9)                590
Fresh Papaya(1-4)                590
Fresh Tomato(1-10)               590
Fresh eggplant(1-4)              590
Fresh pineapple(1-15)            590
Rotten Bittermelon(5-8)          590
Rotten Cucumber(12-20)           590
Rotten Orange(20-35)             590
Rotten Papaya(7-12)              590
Rotten Pineapple(25-35)          590
Rotten Tomato(24-35)             590
Rotten banana(7-13)              590
Rotten eggplant(8-15)            590
Semi Fresh Bittermelon ( 3-5)    590
Semi Fresh Cucumber(6-12)        590
Semi Fresh Papaya(4-7)           590
Semi fresh Orange(9-20)          590
Semi fresh Pineapple (15-25)     590
Semi fresh Tomato(10-24)         590
Semi fresh banana(4-7)           590
Semi_Fresh eggplant(4-8)         590


## 3. Cleaning and duplicate audit

Files prefixed with `aug_<number>_` are excluded because the publisher has already augmented them. All remaining images are fully decoded, EXIF orientation is corrected, and pixels are converted to RGB. Exact duplicates are detected from decoded pixels and dimensions, so differences in file metadata do not hide duplicates.

If identical pixels have conflicting class labels, every image in that group is excluded rather than guessing a label. Same-label duplicates retain one deterministic representative. Images with unusual dimensions are recorded, and legitimate dark or blurred images are retained because automatic cutoffs could remove real spoilage examples.


In [ ]:
audit = []
for index, name in enumerate(entries):
    folder = Path(name).parent.name
    row = {"sample_id": f"img_{index:05d}", "source_path": name,
           **labels[folder], "class_id": class_to_id[labels[folder]["class_name"]]}
    row["is_publisher_augmentation"] = bool(re.match(r"^aug_\d+_", Path(name).name, re.I))
    if row["is_publisher_augmentation"]:
        row["status"] = "excluded_publisher_augmentation"
    else:
        try:
            with Image.open(BytesIO(archive.read(name))) as image:
                image.load()
                rgb = ImageOps.exif_transpose(image).convert("RGB")
                row.update({"width": rgb.width, "height": rgb.height,
                            "source_mode": image.mode,
                            "pixel_sha256": hashlib.sha256(
                                f"{rgb.width}x{rgb.height}:".encode() + rgb.tobytes()).hexdigest(),
                            "status": "candidate"})
        except (OSError, ValueError, Image.DecompressionBombError) as error:
            row.update({"status": "excluded_unreadable", "error": str(error)})
    audit.append(row)
audit = pd.DataFrame(audit)
candidates = audit[audit.status.eq("candidate")]
for digest, group in candidates.groupby("pixel_sha256", sort=True):
    if group.class_name.nunique() > 1:
        audit.loc[group.index, "status"] = "excluded_conflicting_labels"
    else:
        audit.loc[group.index[0], "status"] = "kept"
        audit.loc[group.index[1:], "status"] = "excluded_exact_duplicate"
kept = audit[audit.status.eq("kept")].copy()
assert len(kept), "No usable images remained."
assert kept.pixel_sha256.is_unique
print(audit.status.value_counts().to_string())
print("Candidate dimension counts:")
print(candidates.groupby(["width", "height"]).size().to_string())


status
excluded_publisher_augmentation    8943
kept                               5174
excluded_conflicting_labels          34
excluded_exact_duplicate              9
Candidate dimension counts:
width  height
512.0  512.0     5217


## 4. Image preparation and feature extraction

Each retained image is fitted inside 224 by 224 pixels without stretching, then centered on a black canvas and saved as a lossless PNG. Numerical features are calculated from the resized content before padding, so padding does not affect the measurements. They include RGB and HSV means and standard deviations, 16-bin RGB histograms, grayscale brightness and entropy, and edge and Laplacian measurements.

RGB, saturation, value, and brightness statistics use pixel values divided by 255. Hue is represented by sine and cosine means because it is circular. These features provide a tabular ML input; further scaling must be fitted on training data only. Features can include background effects because no produce segmentation is performed.


In [ ]:
def extract_features(content):
    rgb = np.asarray(content, dtype=np.float32) / 255.0
    hsv = np.asarray(content.convert("HSV"), dtype=np.float32) / 255.0
    features = {}
    for i, channel in enumerate("rgb"):
        features[f"{channel}_mean"] = float(rgb[..., i].mean())
        features[f"{channel}_std"] = float(rgb[..., i].std())
        counts, _ = np.histogram(rgb[..., i], bins=16, range=(0, 1))
        for bin_id, value in enumerate(counts / counts.sum()):
            features[f"{channel}_hist_{bin_id:02d}"] = float(value)
    angle = hsv[..., 0] * 2 * np.pi
    features["hue_sin_mean"] = float(np.sin(angle).mean())
    features["hue_cos_mean"] = float(np.cos(angle).mean())
    for i, channel in [(1, "saturation"), (2, "value")]:
        features[f"{channel}_mean"] = float(hsv[..., i].mean())
        features[f"{channel}_std"] = float(hsv[..., i].std())
    gray = np.asarray(content.convert("L"), dtype=np.float32) / 255.0
    features["brightness_mean"] = float(gray.mean())
    features["brightness_std"] = float(gray.std())
    p = np.bincount(np.asarray(content.convert("L")).ravel(), minlength=256).astype(float)
    p = p[p > 0] / p.sum()
    features["gray_entropy_bits"] = float(-(p * np.log2(p)).sum())
    if min(gray.shape) >= 3:
        center = gray[1:-1, 1:-1]
        lap = gray[:-2, 1:-1] + gray[2:, 1:-1] + gray[1:-1, :-2] + gray[1:-1, 2:] - 4 * center
        features["laplacian_variance"] = float(lap.var())
        features["edge_strength_mean"] = float((np.abs(gray[1:, :-1] - gray[:-1, :-1]) + np.abs(gray[:-1, 1:] - gray[:-1, :-1])).mean())
    else:
        features["laplacian_variance"] = 0.0
        features["edge_strength_mean"] = 0.0
    return features

manifest_rows, feature_rows = [], []
for count, (idx, row) in enumerate(kept.iterrows(), 1):
    with Image.open(BytesIO(archive.read(row.source_path))) as image:
        rgb = ImageOps.exif_transpose(image).convert("RGB")
        content = ImageOps.contain(rgb, (224, 224), Image.Resampling.LANCZOS)
        canvas = Image.new("RGB", (224, 224), (0, 0, 0))
        canvas.paste(content, ((224 - content.width) // 2, (224 - content.height) // 2))
        relative_path = f"images/{row.class_name}/{row.sample_id}.png"
        destination = OUT / relative_path
        destination.parent.mkdir(parents=True, exist_ok=True)
        canvas.save(destination)
        metadata = row.to_dict()
        metadata.update({"image_path": relative_path,
                         "content_width": content.width, "content_height": content.height})
        manifest_rows.append(metadata)
        feature_rows.append({"sample_id": row.sample_id, **extract_features(content)})
        audit.loc[idx, "image_path"] = relative_path
    if count % 1000 == 0:
        print("Prepared", count, "images")
manifest = pd.DataFrame(manifest_rows)
features = pd.DataFrame(feature_rows)
manifest.to_csv(OUT / "manifest.csv", index=False)
features.to_csv(OUT / "features.csv", index=False)
audit.to_csv(OUT / "cleaning_audit.csv", index=False)
class_counts = manifest.groupby(["class_id", "class_name", "produce", "freshness"]).size().reset_index(name="image_count")
class_counts.to_csv(OUT / "class_counts.csv", index=False)
print("Prepared images:", len(manifest), "Numerical features:", features.shape[1] - 1)
print(class_counts.to_string(index=False))


Prepared 1000 images


Prepared 2000 images


Prepared 3000 images


Prepared 4000 images


Prepared 5000 images


Prepared images: 5174 Numerical features: 65
 class_id             class_name     produce  freshness  image_count
        0           fresh_banana      banana      fresh          173
        1      fresh_bittermelon bittermelon      fresh          196
        2         fresh_cucumber    cucumber      fresh          140
        3         fresh_eggplant    eggplant      fresh          236
        4           fresh_orange      orange      fresh          238
        5           fresh_papaya      papaya      fresh          118
        6        fresh_pineapple   pineapple      fresh          220
        7           fresh_tomato      tomato      fresh          590
        8          rotten_banana      banana     rotten          301
        9     rotten_bittermelon bittermelon     rotten          214
       10        rotten_cucumber    cucumber     rotten           47
       11        rotten_eggplant    eggplant     rotten          153
       12          rotten_orange      orange     rotten   

## 5. Validate the prepared dataset

Validation checks every saved image, its size and mode, feature finiteness, sample alignment, and duplicate removal. The audit includes every source image and gives a reason for each exclusion. Exact duplicate checks do not prove that retained images show different physical produce items; the publisher does not supply reliable subject or capture-session IDs.


In [ ]:
assert len(audit) == len(entries)
assert len(manifest) == len(features)
assert manifest.sample_id.is_unique and features.sample_id.is_unique
assert manifest.sample_id.tolist() == features.sample_id.tolist()
assert manifest.pixel_sha256.is_unique
assert not manifest.is_publisher_augmentation.any()
assert np.isfinite(features.drop(columns="sample_id").to_numpy()).all()
assert manifest[["class_id", "class_name", "produce", "freshness", "image_path"]].notna().all().all()
for path in manifest.image_path:
    with Image.open(OUT / path) as image:
        image.load()
        assert image.size == (224, 224) and image.mode == "RGB"
print("All prepared images and feature rows passed validation.")
summary = {"dataset": "AgriFreshNET version 1", "doi": "10.17632/42m5tb7yv9.1",
           "source_url": "https://data.mendeley.com/datasets/42m5tb7yv9/1",
           "source_archive": "Processed Data.zip", "source_sha256": archive_hash,
           "source_is_raw": False, "raw_archive_published": False,
           "source_image_count": len(entries), "retained_image_count": len(manifest),
           "source_class_count": len(class_names), "retained_class_count": manifest.class_name.nunique(),
           "numerical_feature_count": features.shape[1] - 1,
           "status_counts": {str(k): int(v) for k, v in audit.status.value_counts().items()},
           "prepared_image_size": [224, 224], "validation_passed": True,
           "limitations": ["Source is already processed; strict raw-data requirement is unmet.",
                           "Folder labels are inherited, not visually validated.",
                           "Day intervals are publisher metadata, not measured remaining shelf life.",
                           "No verified subject IDs; near duplicates may remain.",
                           "Features include background; no produce segmentation."]}
(OUT / "preparation_summary.json").write_text(json.dumps(summary, indent=2))
print(json.dumps(summary, indent=2))


All prepared images and feature rows passed validation.
{
  "dataset": "AgriFreshNET version 1",
  "doi": "10.17632/42m5tb7yv9.1",
  "source_url": "https://data.mendeley.com/datasets/42m5tb7yv9/1",
  "source_archive": "Processed Data.zip",
  "source_sha256": "6607a48bf2817f9dfc8d4a1c32ce663c1d8308b4b1251e9bf65f074fcac53f11",
  "source_is_raw": false,
  "raw_archive_published": false,
  "source_image_count": 14160,
  "retained_image_count": 5174,
  "source_class_count": 24,
  "retained_class_count": 24,
  "numerical_feature_count": 65,
  "status_counts": {
    "excluded_publisher_augmentation": 8943,
    "kept": 5174,
    "excluded_conflicting_labels": 34,
    "excluded_exact_duplicate": 9
  },
  "prepared_image_size": [
    224,
    224
  ],
  "validation_passed": true,
  "limitations": [
    "Source is already processed; strict raw-data requirement is unmet.",
    "Folder labels are inherited, not visually validated.",
    "Day intervals are publisher metadata, not measured remaining 

## 6. Load the data for a later ML task



In [ ]:
saved_manifest = pd.read_csv(OUT / "manifest.csv")
saved_features = pd.read_csv(OUT / "features.csv")
ml_table = saved_manifest[["sample_id", "class_id", "freshness"]].merge(
    saved_features, on="sample_id", validate="one_to_one")
feature_columns = [c for c in saved_features.columns if c != "sample_id"]
X = ml_table[feature_columns].to_numpy(dtype=np.float32)
y = ml_table["class_id"].to_numpy(dtype=np.int64)
def load_normalized_image(image_path):
    with Image.open(OUT / image_path) as image:
        return np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
example = load_normalized_image(saved_manifest.iloc[0].image_path)
assert example.shape == (224, 224, 3)
assert 0 <= example.min() <= example.max() <= 1
print("Feature matrix:", X.shape, "Target:", y.shape)
print("Normalized image shape:", example.shape, "dtype:", example.dtype,
      "range:", float(example.min()), float(example.max()))
archive.close()


Feature matrix: (5174, 65) Target: (5174,)
Normalized image shape: (224, 224, 3) dtype: float32 range: 0.0 1.0


## 7. Download the prepared dataset

Run this cell on its own to download FreshPick_Prepared_Dataset.zip from Google Drive. Connect the Google account holding the FreshPick_Submission folder when prompted; no preparation rerun is needed.

[Submission folder: prepared dataset and notebook](https://drive.google.com/drive/folders/1oTyV8cB-Rq9znji36Y4E3OU3Ec_DDv-e).


In [ ]:
from pathlib import Path
import hashlib
import shutil
import zipfile

EXPECTED_ZIP_SHA256 = "5a92454c89393e20bc596ecdd2d07d676f161358d96682b43a3ff169cd826c54"
ZIP_NAME = "FreshPick_Prepared_Dataset.zip"
try:
    from google.colab import drive, files
except ImportError:
    in_colab = False
    download_zip = Path.cwd() / ZIP_NAME
else:
    in_colab = True
    download_zip = Path("/content") / ZIP_NAME
    if not download_zip.is_file():
        print("Connect the Google account containing the FreshPick_Submission folder.")
        drive.mount("/content/drive")
        drive_zip = Path("/content/drive/MyDrive/FreshPick_Submission") / ZIP_NAME
        if not drive_zip.is_file():
            raise FileNotFoundError("Open the FreshPick_Submission folder using the same Google account: https://drive.google.com/drive/folders/1oTyV8cB-Rq9znji36Y4E3OU3Ec_DDv-e")
        print("Copying the prepared dataset from Google Drive...")
        shutil.copyfile(drive_zip, download_zip)
if not download_zip.is_file():
    raise FileNotFoundError(f"Place {ZIP_NAME} beside this notebook, or run the cell in Google Colab.")
digest = hashlib.sha256()
with download_zip.open("rb") as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(block)
assert digest.hexdigest() == EXPECTED_ZIP_SHA256, "Dataset checksum mismatch."
with zipfile.ZipFile(download_zip) as archive:
    assert archive.testzip() is None, "Dataset ZIP integrity check failed."
print(f"Prepared dataset verified: {download_zip.stat().st_size / 1e6:.1f} MB")
if in_colab:
    files.download(str(download_zip))
else:
    print(f"Download the dataset ZIP from: {download_zip}")
